# 第9章: 事前学習済み言語モデル（BERT型）


In [10]:
# OpenMPライブラリの二重読み込みによるカーネルのクラッシュを防ぐ（torch より前に設定）
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# ---- import ----
from itertools import combinations
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
)

MODEL_NAME = "bert-base-uncased"     # BERT型の事前学習済みモデル
DATA_DIR = "../chapter 7/SST-2"      # 第7章のポジネガ分類データ

# トークナイザの読み込み
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

## 80. トークン化

与えられた文を BERT のトークナイザでトークンに分解し、トークン列を表示する。

In [11]:
text = "The movie was full of incomprehensibilities."

# サブワード単位でトークンに分解する
tokens = tokenizer.tokenize(text)
print(tokens)

['the', 'movie', 'was', 'full', 'of', 'inc', '##omp', '##re', '##hen', '##si', '##bilities', '.']


## 81. マスクの予測

`"The movie was full of [MASK]."` の `[MASK]` を埋めるのに最も適切なトークンを求める。

In [12]:
# マスク言語モデルの読み込み
mlm = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)
mlm.eval()

text = "The movie was full of [MASK]."
inputs = tokenizer(text, return_tensors="pt")
# [MASK] トークンの位置を特定する
mask_pos = (inputs["input_ids"][0] == tokenizer.mask_token_id).nonzero().item()

with torch.no_grad():
    logits = mlm(**inputs).logits

# マスク位置で最もスコアの高いトークンを取り出す
best_id = logits[0, mask_pos].argmax().item()
print(tokenizer.decode([best_id]))

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 22291.34it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


fun


## 82. マスクのtop-k予測

`[MASK]` に埋めるのに適切なトークンの上位10個と、その確率（尤度）を求める。

In [13]:
# マスク位置のスコアを確率に変換し、上位10件を取り出す
probs = logits[0, mask_pos].softmax(dim=-1)
topk = probs.topk(10)
for score, idx in zip(topk.values, topk.indices):
    print(f"{tokenizer.decode([idx]):15s} {score.item():.4f}")

fun             0.1071
surprises       0.0663
drama           0.0447
stars           0.0272
laughs          0.0254
action          0.0195
excitement      0.0190
people          0.0183
tension         0.0150
music           0.0146


## 83. CLSトークンによる文ベクトル

各文の全ての組み合わせについて、最終層の `[CLS]` トークンの埋め込みベクトルを
用いてコサイン類似度を求める。

In [14]:
# 隠れ状態を取り出すためのモデル
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

sentences = [
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish.",
]

def cls_vector(text):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        out = model(**inputs)
    return out.last_hidden_state[0, 0]   # 先頭(位置0)が [CLS]

vecs = {s: cls_vector(s) for s in sentences}
for a, b in combinations(sentences, 2):   # 全ての2文の組み合わせ
    cos = torch.cosine_similarity(vecs[a], vecs[b], dim=0).item()
    print(f"{cos:.4f}  |  {a}  <->  {b}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16537.87it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.9881  |  The movie was full of fun.  <->  The movie was full of excitement.
0.9558  |  The movie was full of fun.  <->  The movie was full of crap.
0.9475  |  The movie was full of fun.  <->  The movie was full of rubbish.
0.9541  |  The movie was full of excitement.  <->  The movie was full of crap.
0.9487  |  The movie was full of excitement.  <->  The movie was full of rubbish.
0.9807  |  The movie was full of crap.  <->  The movie was full of rubbish.


## 84. 平均による文ベクトル

各文の全ての組み合わせについて、最終層の埋め込みベクトルの平均を
用いてコサイン類似度を求める。

In [15]:
def mean_vector(text):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        out = model(**inputs)
    hidden = out.last_hidden_state[0]                 # (seq, dim)
    mask = inputs["attention_mask"][0].unsqueeze(-1)  # パディング除外用
    return (hidden * mask).sum(0) / mask.sum()        # トークン方向の平均

vecs = {s: mean_vector(s) for s in sentences}
for a, b in combinations(sentences, 2):
    cos = torch.cosine_similarity(vecs[a], vecs[b], dim=0).item()
    print(f"{cos:.4f}  |  {a}  <->  {b}")

0.9568  |  The movie was full of fun.  <->  The movie was full of excitement.
0.8490  |  The movie was full of fun.  <->  The movie was full of crap.
0.8169  |  The movie was full of fun.  <->  The movie was full of rubbish.
0.8352  |  The movie was full of excitement.  <->  The movie was full of crap.
0.7938  |  The movie was full of excitement.  <->  The movie was full of rubbish.
0.9226  |  The movie was full of crap.  <->  The movie was full of rubbish.


## 85. データセットの準備

訓練セット (`train.tsv`) と開発セット (`dev.tsv`) のテキストと極性ラベルを読み込み、
全てのテキストをトークン列に変換する。

In [16]:
def load_dataset(path):
    data = []
    with open(path, encoding="utf-8") as f:
        next(f)  # ヘッダをスキップ
        for line in f:
            text, label = line.rstrip("\n").rsplit("\t", 1)
            input_ids = tokenizer(text)["input_ids"]   # 特殊トークン付きのトークン列
            data.append({"text": text, "label": int(label), "input_ids": input_ids})
    return data

train = load_dataset(f"{DATA_DIR}/train.tsv")
dev   = load_dataset(f"{DATA_DIR}/dev.tsv")

## 86. ミニバッチの作成

訓練データの冒頭4事例に対してパディングを行い、トークン列の長さを揃えて
ミニバッチを構成する。

In [17]:
batch = train[:4]

# 最長に合わせてパディングし、テンソルにまとめる
enc = tokenizer(
    [ex["text"] for ex in batch],
    padding=True,
    return_tensors="pt",
)
enc["labels"] = torch.tensor([ex["label"] for ex in batch])

print(enc["input_ids"].shape)   # (4, 最長トークン長)
print(enc["input_ids"])

torch.Size([4, 15])
tensor([[  101,  5342,  2047,  3595,  8496,  2013,  1996, 18643,  3197,   102,
             0,     0,     0,     0,     0],
        [  101,  3397,  2053, 15966,  1010,  2069,  4450,  2098, 18201,  2015,
           102,     0,     0,     0,     0],
        [  101,  2008,  7459,  2049,  3494,  1998, 10639,  2015,  2242,  2738,
          3376,  2055,  2529,  3267,   102],
        [  101,  3464, 12580,  8510,  2000,  3961,  1996,  2168,  2802,   102,
             0,     0,     0,     0,     0]])


## 87. ファインチューニング

訓練セットを用いて事前学習済みモデルを極性分析タスク向けにファインチューニングし、
開発セット上での正解率を計測する。

In [18]:
# cuda -> mps(Apple GPU) -> cpu の順に利用可能なデバイスを選ぶ
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available()
          else "cpu")

# 分類ヘッド付きモデル（2クラス）
clf = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

def collate(batch):
    # バッチ内のテキストをまとめてトークン化・パディングする
    enc = tokenizer(
        [ex["text"] for ex in batch],
        padding=True, truncation=True, return_tensors="pt",
    )
    enc["labels"] = torch.tensor([ex["label"] for ex in batch])
    return enc

train_loader = DataLoader(train, batch_size=16, shuffle=True, collate_fn=collate)
dev_loader   = DataLoader(dev, batch_size=64, collate_fn=collate)
optimizer = torch.optim.AdamW(clf.parameters(), lr=2e-5)

for epoch in range(2):
    clf.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = clf(**batch)          # labels を渡すと loss も計算される
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    print(f"epoch {epoch + 1}: loss = {total_loss / len(train_loader):.4f}")

def accuracy(model, loader):
    model.eval()
    correct = n = 0
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            pred = model(**batch).logits.argmax(dim=-1)
            correct += (pred == batch["labels"]).sum().item()
            n += len(pred)
    return correct / n

print(f"開発セット正解率: {accuracy(clf, dev_loader):.4f}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15133.93it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

epoch 1: loss = 0.2011
epoch 2: loss = 0.1072
開発セット正解率: 0.9278


## 88. 極性分析

問題87でファインチューニングしたモデルを用いて、各文の極性を予測する。

In [19]:
texts = [
    "The movie was full of incomprehensibilities.",
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish.",
]

clf.eval()
enc = tokenizer(texts, padding=True, return_tensors="pt").to(device)
with torch.no_grad():
    preds = clf(**enc).logits.argmax(dim=-1)

for text, pred in zip(texts, preds):
    label = "ポジティブ" if pred.item() == 1 else "ネガティブ"
    print(f"{label}: {text}")

ネガティブ: The movie was full of incomprehensibilities.
ポジティブ: The movie was full of fun.
ポジティブ: The movie was full of excitement.
ネガティブ: The movie was full of crap.
ネガティブ: The movie was full of rubbish.


## 89. アーキテクチャの変更

問題87とは異なるアーキテクチャの分類モデルを設計してファインチューニングし、
開発セット上での正解率を計測する。ここでは各トークンの最大値プーリングを用いる。

In [20]:
# BERT の出力トークン列を最大値プーリングして分類するモデル
class BertMaxPoolClassifier(nn.Module):
    def __init__(self, model_name, num_labels=2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden = out.last_hidden_state                       # (batch, seq, dim)
        mask = attention_mask.unsqueeze(-1).bool()
        hidden = hidden.masked_fill(~mask, float("-inf"))    # パディングを除外
        pooled = hidden.max(dim=1).values                    # 各次元の最大値プーリング
        logits = self.fc(pooled)
        loss = None
        if labels is not None:
            loss = nn.functional.cross_entropy(logits, labels)
        return logits, loss

# cuda -> mps(Apple GPU) -> cpu の順に利用可能なデバイスを選ぶ
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available()
          else "cpu")
model2 = BertMaxPoolClassifier(MODEL_NAME).to(device)
optimizer = torch.optim.AdamW(model2.parameters(), lr=2e-5)

for epoch in range(2):
    model2.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        _, loss = model2(**batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch {epoch + 1}: loss = {total_loss / len(train_loader):.4f}")

def accuracy2(model, loader):
    model.eval()
    correct = n = 0
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits, _ = model(**batch)
            pred = logits.argmax(dim=-1)
            correct += (pred == batch["labels"]).sum().item()
            n += len(pred)
    return correct / n

print(f"開発セット正解率: {accuracy2(model2, dev_loader):.4f}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9945.50it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 1: loss = 0.1999
epoch 2: loss = 0.1038
開発セット正解率: 0.9220
